In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/projet.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
df = con.execute("SELECT * FROM donnees ORDER BY debut_utc").df()
df["annee"] = df["debut_utc"].dt.tz_convert("America/Toronto").dt.year

In [2]:
# Prévision = la demande 168 heures avant (même heure, même jour, semaine passée)
df["prevision_base"] = df["demande_mw"].shift(168)

test = df[df["annee"] == 2023].dropna(subset=["demande_mw", "prevision_base"])
erreur = (test["demande_mw"] - test["prevision_base"]).abs()

mae = erreur.mean()
mape = (erreur / test["demande_mw"]).mean() * 100
print(f"Méthode de base sur 2023 : erreur moyenne {mae:.0f} MW, soit {mape:.1f} %")

Méthode de base sur 2023 : erreur moyenne 1763 MW, soit 7.4 %


In [4]:
import holidays

locale = df["debut_utc"].dt.tz_convert("America/Toronto")
feries = holidays.Canada(prov="QC", years=range(2019, 2025))

df["chauffage"] = (18 - df["temp_ponderee"]).clip(lower=0)      # degrés de froid
df["climatisation"] = (df["temp_ponderee"] - 18).clip(lower=0)  # degrés de chaleur
df["heure"] = locale.dt.hour
df["jour_semaine"] = locale.dt.dayofweek
df["ferie"] = locale.dt.date.map(lambda d: d in feries).astype(int)
df["tendance"] = df["annee"] - 2019

In [6]:
from sklearn.linear_model import LinearRegression

variables = ["chauffage", "climatisation", "ferie", "tendance"]
X = pd.get_dummies(df[variables + ["heure", "jour_semaine"]],
                   columns=["heure", "jour_semaine"], drop_first=True)
y = df["demande_mw"]

ok = y.notna() & X.notna().all(axis=1)
entrainement = ok & (df["annee"] <= 2022)
test_2023 = ok & (df["annee"] == 2023)

modele = LinearRegression().fit(X[entrainement], y[entrainement])
prevision = modele.predict(X[test_2023])

erreur = (y[test_2023] - prevision).abs()
print(f"Régression sur 2023 : erreur moyenne {erreur.mean():.0f} MW, "
      f"soit {(erreur / y[test_2023]).mean() * 100:.1f} %")

Régression sur 2023 : erreur moyenne 1061 MW, soit 5.2 %


In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor

variables_gb = ["chauffage", "climatisation", "temp_ponderee", "ferie",
                "tendance", "heure", "jour_semaine"]
Xg = df[variables_gb]

gb = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, random_state=0)
gb.fit(Xg[entrainement], y[entrainement])
prevision_gb = gb.predict(Xg[test_2023])

erreur_gb = (y[test_2023] - prevision_gb).abs()
print(f"Gradient boosting sur 2023 : erreur moyenne {erreur_gb.mean():.0f} MW, "
      f"soit {(erreur_gb / y[test_2023]).mean() * 100:.1f} %")

Gradient boosting sur 2023 : erreur moyenne 740 MW, soit 3.3 %
